# GenCAD baseline: untuned Qwen2.5-VL-3B, image -> CadQuery
MecAgent technical test (`CADCODER/GenCAD-Code`), scored with MecAgent's own metric code
(`metrics/best_iou.py`, `metrics/valid_syntax_rate.py`) cloned from their repo.

Run top to bottom on a T4 GPU runtime. Generations are saved to Google Drive after every
sample, so if Colab disconnects, reconnect and run all cells again: it resumes where it stopped.

Set `MODEL_TAG` / `ADAPTER` in cell 3 to score a fine-tuned model later with the same code.

In [ ]:
# 1 - installs, MecAgent metrics, Drive
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cadquery", "trimesh", "unsloth"], check=True)
subprocess.run(["git", "clone", "-q", "https://github.com/MecAgent/mecagent-technical-test", "/content/mecagent-technical-test"])
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/gencad"
os.makedirs(f"{OUT}/steps", exist_ok=True)
print("ok, outputs go to", OUT)


In [ ]:
# 2 - held-out eval set (fixed, saved to Drive so base and tuned use the same rows)
# Eval set = the dataset's own `hundred_subset` (exactly 100 test rows), not a random pick.
import json, glob
from datasets import load_dataset
test = load_dataset("CADCODER/GenCAD-Code", split="test")
eval_idx = [i for i, f in enumerate(test["hundred_subset"]) if f]
print("test rows:", len(test), "| eval set (hundred_subset):", len(eval_idx), "rows, first ids", eval_idx[:5])

# If Drive holds results for a different eval set (the earlier random-100 draft), clear them.
ids_path = f"{OUT}/eval_ids.json"
if os.path.exists(ids_path) and json.load(open(ids_path)) != eval_idx:
    for f in [ids_path] + glob.glob(f"{OUT}/gen_*.jsonl") + glob.glob(f"{OUT}/scores_*.json"):
        os.remove(f); print("removed stale", f)
json.dump(eval_idx, open(ids_path, "w"))

prompts = {test[i]["prompt"] for i in eval_idx}
print("distinct prompts in eval set:", len(prompts))
PROMPT = test[eval_idx[0]]["prompt"]
print("prompt:", PROMPT)


In [ ]:
# 3 - model (untuned base)
import unsloth  # must be imported before transformers
from unsloth import FastVisionModel
import torch

MODEL_TAG = "qwen25vl3b_base"      # later: "qwen25vl3b_qlora"
BASE = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"
ADAPTER = None                      # later: f"{OUT}/qwen_lora_adapter"

model, tokenizer = FastVisionModel.from_pretrained(ADAPTER or BASE, load_in_4bit=True)
FastVisionModel.for_inference(model)

INSTRUCTION = PROMPT  # the dataset's own prompt; fine-tuning uses the same one

# How long are the reference programs? Sets the generation budget.
import numpy as np
gt_len = [len(tokenizer.tokenizer(test[i]["cadquery"])["input_ids"]) for i in eval_idx]
print("reference code tokens: p50 %d  p90 %d  p95 %d  max %d" % (
    np.percentile(gt_len, 50), np.percentile(gt_len, 90), np.percentile(gt_len, 95), max(gt_len)))
MAX_NEW = int(min(2048, max(512, np.percentile(gt_len, 95) * 1.2)))
print("max_new_tokens =", MAX_NEW)


In [ ]:
# 4 - generate (resumable, ~1 min per sample on a T4)
import re, time

def strip_fences(text):
    m = re.search(r"```[\w+-]*[ \t]*\n(.*?)(```|$)", text, re.S)  # any language tag
    return m.group(1) if m else text

def generate(image):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": INSTRUCTION}]}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image.convert("RGB"), prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

gen_path = f"{OUT}/gen_{MODEL_TAG}.jsonl"
done = set()
if os.path.exists(gen_path):
    done = {json.loads(l)["idx"] for l in open(gen_path)}
print(f"{len(done)}/{len(eval_idx)} already generated")

t0 = time.time()
for n, i in enumerate(eval_idx):
    if i in done:
        continue
    raw = generate(test[i]["image"])
    with open(gen_path, "a") as f:
        f.write(json.dumps({"idx": i, "raw": raw, "code": strip_fences(raw)}) + "\n")
    print(f"[{n+1}/{len(eval_idx)}] idx {i}  {time.time()-t0:.0f}s elapsed")


In [ ]:
# 5 - score with MecAgent's metric code (each sample in its own process, 120 s timeout)
SCORER = r'''
import sys, json
sys.path.insert(0, "/content/mecagent-technical-test")
from metrics.best_iou import _load_solid_from_code, _normalized_mesh, iou_best
import cadquery as cq
gt, pr, step_out = open(sys.argv[1]).read(), open(sys.argv[2]).read(), sys.argv[3]
r = {"valid": False, "iou": 0.0, "err": None}
try:
    s_pr = _load_solid_from_code(pr, "pred")
    r["valid"] = True
except Exception as e:
    r["err"] = str(e)[:300]
if r["valid"]:
    try:
        cq.exporters.export(s_pr, step_out)
        s_gt = _load_solid_from_code(gt, "gt")
        r["iou"] = float(iou_best(_normalized_mesh(s_gt), _normalized_mesh(s_pr)))
    except Exception as e:
        r["err"] = "iou: " + str(e)[:300]
print(json.dumps(r))
'''
open("/content/score_one.py", "w").write(SCORER)

rows = [json.loads(l) for l in open(gen_path)]
results = []
for row in rows:
    i = row["idx"]
    open("/tmp/gt.py", "w").write(test[i]["cadquery"])
    open("/tmp/pr.py", "w").write(row["code"])
    step = f"{OUT}/steps/{MODEL_TAG}_{i}.step"
    try:
        p = subprocess.run([sys.executable, "/content/score_one.py", "/tmp/gt.py", "/tmp/pr.py", step],
                           capture_output=True, text=True, timeout=120)
        r = json.loads(p.stdout.strip().splitlines()[-1]) if p.stdout.strip() else \
            {"valid": False, "iou": 0.0, "err": "crash: " + p.stderr[-300:]}
    except subprocess.TimeoutExpired:
        r = {"valid": False, "iou": 0.0, "err": "timeout 120s"}
    r["idx"] = i
    results.append(r)

json.dump(results, open(f"{OUT}/scores_{MODEL_TAG}.json", "w"), indent=1)
n = len(results)
valid = [r for r in results if r["valid"]]
vsr = len(valid) / n
iou_valid = np.mean([r["iou"] for r in valid]) if valid else 0.0   # MecAgent's definition
iou_all = np.mean([r["iou"] for r in results])                    # invalid counted as 0
print(f"{MODEL_TAG}: n={n}  VSR={vsr:.3f}  IoU_best(valid only, MecAgent)={iou_valid:.3f}  IoU_best(invalid=0)={iou_all:.3f}")


In [ ]:
# 6 - look at a few examples (for the walkthrough: good ones AND failures)
from collections import Counter
print("failure kinds:", Counter((r["err"] or "ok").split(":")[0][:40] for r in results).most_common(8))
for r in sorted(results, key=lambda r: -r["iou"])[:2] + [r for r in results if not r["valid"]][:2]:
    row = next(x for x in rows if x["idx"] == r["idx"])
    print("=" * 80, f"\nidx {r['idx']}  valid={r['valid']}  iou={r['iou']:.3f}  err={r['err']}")
    print(row["code"][:600])
